# **LAB 4: LLS and Prompt Engineering for Decision Support**

### Part 0: Repository and API-key Setup

In [13]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

from dotenv import load_dotenv
load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]


# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


## **Section 1 - Talking to an LLM Programmatically**

## Part 1.1 - Your first API call

In [14]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
# def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
#             temperature=0.7, max_tokens=500):
#     response = client.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "system", "content": system_prompt},
#             {"role": "user",   "content": user_prompt},
#         ],
#         temperature=temperature,
#         max_tokens=max_tokens,
#     )
#     return response.choices[0].message.content

def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content, response.usage


# TODO: Call it once with a simple question and print the answer.
# TODO: Print response.usage as well — how many tokens did your call consume?
answer, usage  = ask_llm("What is microfinance in one sentence?")
print(answer)
print(usage)

Microfinance refers to the provision of small loans, savings, and other financial services to low-income individuals or groups who lack access to traditional banking services, helping them to start or expand small businesses and improve their economic well-being.
CompletionUsage(completion_tokens=46, prompt_tokens=49, total_tokens=95, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.041572737, prompt_time=0.006910637, completion_time=0.170857958, total_time=0.177768595)


**Anatomy of a call**

1. What is the difference between the system and user roles? Give an example of something that belongs in each.


2. What is a token, roughly? Why do API providers bill per token rather than per request?

## Part 1.2 - Temperature: the randomness dial

In [15]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
question = "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.
print('===== Temperature: 0.0 ======')
for i in range(5):
    answer = ask_llm(question, temperature = 0.0)
    print(f"Run {i+1}: {answer}")

print()
print('===== Temperature: 1.2 =====')
for i in range(5):
    answer = ask_llm(question, temperature = 1.2)
    print(f"Run {i+1}: {answer}")

===== Temperature: 0.0 ======
Run 1: ('Here are a few suggestions for a savings product for market traders in Accra:\n\n1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.\n2. **Trader\'s Treasure**: This name emphasizes the idea of saving and accumulating wealth.\n3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", which could appeal to market traders in Accra.\n4. **Market Mobi**: This name incorporates "mobi", short for mobile, to suggest a convenient and accessible savings product.\n5. **Sika Su**: "Sika" is the Ghanaian word for "money", and "Su" means "save" or "keep", making this name straightforward and easy to understand.\n6. **Kokroko Savings**: "Kokroko" is a Ghanaian word for "honest" or "trustworthy", which could convey a sense of reliability and security for market traders.\n7. **Adanfo Account**: "Adanfo" is a Ghanaian word for "friends" or "partners", suggesting a savings product that s

**Temperature** 

1. What did you observe at each temperature? For the loan decision-support system you are about to build, which temperature regime is appropriate, and why?



In [16]:
## **Section 2 - The Dataset: Loan Application Letters**

In [17]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


## **Section 3 - Prompt Engineering for the Decision Support System**

## Part 3.1 - Component 1: Summarization

In [ ]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1 = "Summarize this:"

print("===== L002 V1 =====")
print(ask_llm(LETTERS["L002"], system_prompt=SUMMARY_PROMPT_V1))

print()
print("===== L006 V1 =====")
print(ask_llm(LETTERS["L006"], system_prompt=SUMMARY_PROMPT_V1))

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

SUMMARY_PROMPT_V2 = "You are an assistant to a microfinance loan officer. Summarize loan applications in 3-4 sentences. Be factual and neutral. Do not invent any details not stated in the letter."

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
print()
print("===== L002 V2 =====")
print(ask_llm(f"Summarize this loan application:\n\n{LETTERS['L002']}", system_prompt=SUMMARY_PROMPT_V2, temperature=0))

print("===== L006 V2 =====")
print(ask_llm(f"Summarize this loan application:\n\n{LETTERS['L006']}", system_prompt=SUMMARY_PROMPT_V2, temperature=0))

===== L002 V1 =====
("Kwame Boateng, a commercial driver in Kumasi, is seeking a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but is optimistic it will improve after the festive season. He doesn't have collateral to offer but is asking for help and promises to repay the loan as soon as possible.", CompletionUsage(completion_tokens=78, prompt_tokens=127, total_tokens=205, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.041449495, prompt_time=0.006356248, completion_time=0.2708395, total_time=0.277195748))

===== L006 V1 =====
('Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends\' opinions. He promises to repay the loan within one year, once his businesses are successful, but has no collateral to of

 **Summarization prompts**
 1. What concrete problems did V1's output have that V2 fixed? Quote examples. 
 
 
 2. Why is "no invented details" an essential instruction in this application? What is this failure mode called in the LLM literature?

## Part 3.2 - Component 2: Structured exctraction (JSON)

In [19]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

**Structured Extraction**

1. Why must the few-shot example NOT come from the six letters you are processing? 


2. Why "use null, do not guess" — what did the model do without that instruction?



3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?

## Part 3.3 - Component 3: The decision-support brief